# Lab 2: DDPG & SAC in TensorFlow

**Course:** Deep Reinforcement Learning · **Framework:** TensorFlow 1.x · **Estimated time:** ~60 min

## Learning Objectives

- Run a baseline DDPG experiment on Pendulum-v1, interpret every column in progress.txt, and load the saved policy for test evaluation
- Deliberately destabilize DDPG and compare QVals and TestEpRet curves against the stable baseline
- Run a controlled SAC vs DDPG comparison on HalfCheetah-v2 across 3 seeds each
- Conduct a SAC temperature (α) ablation across α ∈ {0.05, 0.1, 0.2, 0.5} on Hopper-v2
- Implement a custom Gym-compatible continuous-control environment and train SAC on it
- Compare TRPO with train_v_iters=0 vs 80 to observe how value function quality affects policy gradient (Problem Set 2.1)
- Observe and diagnose a silent DDPG bug from degraded learning curves (Problem Set 2.2)

## Setup

Spinning Up's TF implementations require TensorFlow 1.x. Install and verify.

In [ ]:
!pip install tensorflow==1.15 -q
!git clone https://github.com/openai/spinningup.git 2>/dev/null || echo "Already cloned"
%cd spinningup
!pip install -e . -q

In [ ]:
# Verify: should print DDPG training metrics including TestEpRet and QVals
!python -m spinup.run ddpg_tf1 --env Pendulum-v1 --epochs 5

## Exercise 1: Baseline DDPG on Pendulum-v1

Run a full DDPG baseline and understand the output structure before modifying anything.

In [ ]:
from spinup import ddpg_tf1 as ddpg
import gym

ddpg(
    env_fn=lambda: gym.make('Pendulum-v1'),
    ac_kwargs=dict(hidden_sizes=[64, 64]),
    steps_per_epoch=4000,
    epochs=50,
    replay_size=int(1e6),
    gamma=0.99,
    polyak=0.995,
    pi_lr=1e-3,
    q_lr=1e-3,
    batch_size=100,
    start_steps=10000,
    update_after=1000,
    update_every=50,
    act_noise=0.1,
    num_test_episodes=10,
    max_ep_len=200,
    logger_kwargs=dict(output_dir='/tmp/ddpg-pendulum', exp_name='ddpg-pendulum')
)

In [ ]:
# Inspect the progress log
import pandas as pd
df = pd.read_table('/tmp/ddpg-pendulum/progress.txt')
print(df.columns.tolist())
df[['Epoch', 'AverageEpRet', 'TestEpRet', 'QVals', 'LossPi', 'LossQ']].tail(10)

In [ ]:
# Load the saved policy and run test episodes
!python -m spinup.run test_policy /tmp/ddpg-pendulum/ -n 10

**Tasks:**
1. Which columns in `progress.txt` are most informative for diagnosing learning?
2. Does `QVals` grow monotonically? What would divergence look like?
3. Is `TestEpRet` smooth or noisy — and why?

## Exercise 2: Observing DDPG Instability

DDPG is notoriously sensitive to hyperparameters. Deliberately create instability by increasing learning rates and reducing polyak smoothing, then compare against the stable defaults.

In [ ]:
# Stable baseline
ddpg(
    env_fn=lambda: gym.make('HalfCheetah-v2'),
    ac_kwargs=dict(hidden_sizes=[256, 256]),
    epochs=50,
    gamma=0.99,
    polyak=0.995,
    pi_lr=1e-3,
    q_lr=1e-3,
    act_noise=0.1,
    start_steps=10000,
    update_after=1000,
    logger_kwargs=dict(output_dir='/tmp/ddpg-stable', exp_name='ddpg-stable')
)

In [ ]:
# Deliberately unstable: high lr, reduced polyak, less random exploration
ddpg(
    env_fn=lambda: gym.make('HalfCheetah-v2'),
    ac_kwargs=dict(hidden_sizes=[256, 256]),
    epochs=50,
    polyak=0.99,       # less smoothing than default 0.995
    pi_lr=0.01,        # 10x default — unstable
    q_lr=0.01,
    act_noise=0.1,
    start_steps=1000,  # much less random exploration
    update_after=100,
    logger_kwargs=dict(output_dir='/tmp/ddpg-unstable', exp_name='ddpg-unstable')
)

In [ ]:
!python -m spinup.run plot /tmp/ddpg-stable /tmp/ddpg-unstable

**Observations to record:**
- Does `QVals` diverge in the unstable run?
- Does `TestEpRet` crash after an initial peak?
- How does `polyak` smoothing affect stability, and why?

## Exercise 3: SAC vs DDPG on HalfCheetah-v2

Compare SAC and DDPG head-to-head with matched compute budgets across 3 seeds each.

In [ ]:
from spinup import sac_tf1 as sac, ddpg_tf1 as ddpg
import gym

shared = dict(
    env_fn=lambda: gym.make('HalfCheetah-v2'),
    ac_kwargs=dict(hidden_sizes=[256, 256]),
    steps_per_epoch=4000,
    epochs=100,
    start_steps=10000,
    update_after=1000,
    update_every=50,
    batch_size=100,
    num_test_episodes=10,
    max_ep_len=1000,
)

for seed in [0, 10, 20]:
    ddpg(**shared,
         gamma=0.99, polyak=0.995, pi_lr=1e-3, q_lr=1e-3, act_noise=0.1,
         seed=seed,
         logger_kwargs=dict(output_dir=f'/tmp/ddpg-hc-s{seed}', exp_name='ddpg-hc'))

In [ ]:
for seed in [0, 10, 20]:
    sac(**shared,
        gamma=0.99, polyak=0.995, lr=1e-3, alpha=0.2,
        seed=seed,
        logger_kwargs=dict(output_dir=f'/tmp/sac-hc-s{seed}', exp_name='sac-hc'))

In [ ]:
!python -m spinup.run plot \
    /tmp/ddpg-hc-s0 /tmp/ddpg-hc-s10 /tmp/ddpg-hc-s20 \
    /tmp/sac-hc-s0  /tmp/sac-hc-s10  /tmp/sac-hc-s20

**Analysis:**
- Which algorithm achieves higher `TestEpRet` by epoch 100?
- Which has less variance across seeds?
- At approximately which epoch does SAC surpass DDPG's final performance?

## Exercise 4: SAC Temperature Ablation

SAC's entropy temperature $\alpha$ controls the exploration-exploitation tradeoff. Higher $\alpha$ encourages more exploratory policies; lower $\alpha$ converges toward deterministic behavior.

Run an ablation across $\alpha \in \{0.05, 0.1, 0.2, 0.5\}$ on Hopper-v2.

In [ ]:
from spinup.utils.run_utils import ExperimentGrid
from spinup import sac_tf1

eg = ExperimentGrid(name='sac-alpha-sweep')
eg.add('env_name',             'Hopper-v2',          '',     True)
eg.add('seed',                 [0, 10, 20])
eg.add('epochs',               100)
eg.add('alpha',                [0.05, 0.1, 0.2, 0.5], 'alpha')
eg.add('ac_kwargs:hidden_sizes', [(256, 256)],         'hid')

eg.run(sac_tf1, num_cpu=1)

In [ ]:
!python -m spinup.run plot /root/spinningup/data/sac-alpha-sweep

**Discussion:**
- How does very low $\alpha = 0.05$ change behavior vs. high $\alpha = 0.5$?
- Which $\alpha$ converges fastest on Hopper?
- Why might a high $\alpha$ help early in training but hurt final performance?

> **Note on adaptive $\alpha$:** Spinning Up uses a fixed $\alpha$, but the SAC paper also describes an automatic version that adjusts $\alpha$ to maintain a target entropy level. This generally works better in practice and is available in newer implementations.

## Exercise 5: Custom Gym Environment

Create a custom Gym-compatible continuous-control environment and train SAC on it. This exercise demonstrates how easily SAC generalises to new tasks.

In [ ]:
import gym
import numpy as np
from spinup import sac_tf1 as sac

class SimpleReachEnv(gym.Env):
    """2D point-mass reaching task with a dense reward."""

    def __init__(self):
        self.observation_space = gym.spaces.Box(
            low=-2, high=2, shape=(4,), dtype=np.float32
        )  # [x, y, goal_x, goal_y]
        self.action_space = gym.spaces.Box(
            low=-1, high=1, shape=(2,), dtype=np.float32
        )  # [dx, dy]
        self.goal = np.zeros(2)
        self.pos  = np.zeros(2)

    def reset(self):
        self.pos  = np.random.uniform(-1, 1, 2).astype(np.float32)
        self.goal = np.random.uniform(-1, 1, 2).astype(np.float32)
        return np.concatenate([self.pos, self.goal])

    def step(self, action):
        self.pos = np.clip(self.pos + 0.1 * action, -2, 2)
        dist     = np.linalg.norm(self.pos - self.goal)
        reward   = -dist          # dense: negative distance
        done     = bool(dist < 0.05)
        return np.concatenate([self.pos, self.goal]), reward, done, {}


gym.envs.register(id='SimpleReach-v0', entry_point=SimpleReachEnv, max_episode_steps=50)

sac(
    env_fn=lambda: gym.make('SimpleReach-v0'),
    ac_kwargs=dict(hidden_sizes=[64, 64]),
    epochs=50,
    alpha=0.1,
    start_steps=1000,
    update_after=500,
    logger_kwargs=dict(output_dir='/tmp/sac-reach', exp_name='sac-reach')
)

In [ ]:
# Sparse reward variant — does SAC still learn?
class SimpleReachSparseEnv(SimpleReachEnv):
    def step(self, action):
        obs, _, done, info = super().step(action)
        dist   = np.linalg.norm(self.pos - self.goal)
        reward = 1.0 if dist < 0.05 else 0.0
        return obs, reward, done, info

gym.envs.register(id='SimpleReachSparse-v0',
                  entry_point=SimpleReachSparseEnv, max_episode_steps=50)

sac(
    env_fn=lambda: gym.make('SimpleReachSparse-v0'),
    ac_kwargs=dict(hidden_sizes=[64, 64]),
    epochs=100,
    alpha=0.2,
    start_steps=5000,
    update_after=1000,
    logger_kwargs=dict(output_dir='/tmp/sac-reach-sparse', exp_name='sac-reach-sparse')
)

**Tasks:**
- Does average episode length decrease over training (indicating the agent learns to reach the goal faster)?
- Does the sparse reward variant converge? How does it compare to the dense version?

## Problem Set 2.1: Value Function Fitting in TRPO

**Background.** GAE-Lambda depends on the value function baseline V^π to estimate advantages. If V^π is poorly fit, advantage estimates have high variance and policy updates become unreliable.

Compare TRPO with `train_v_iters=0` (no value function fitting) versus `train_v_iters=80` across 3 seeds. This runs **6 experiments** total.

In [ ]:
!python -m spinup.run trpo_tf1 --env Hopper-v2 \
    --train_v_iters[v] 0 80 \
    --exp_name ex2-1 \
    --epochs 250 \
    --steps_per_epoch 4000 \
    --seed 0 10 20 \
    --dt

In [ ]:
# Find and plot the output directory
import glob, os
dirs = sorted(glob.glob('/root/spinningup/data/ex2-1*/'))
print("Experiment directories found:")
for d in dirs:
    print(' ', d)

if dirs:
    dir_str = ' '.join(dirs)
    !python -m spinup.run plot {dir_str}

**Analysis questions:**
- What is the `AverageEpRet` gap between `train_v_iters=0` and `train_v_iters=80` at epoch 250?
- Does the `train_v_iters=0` run make *any* learning progress, or does it flatline?
- Why does a poorly-fit value function hurt the policy gradient so much?

Official solution: https://spinningup.openai.com/en/latest/spinningup/exercise2_1_soln.html

## Problem Set 2.2: Silent Bug in DDPG

**Key lesson:** failures in RL are frequently *silent* — code runs without errors, but the agent never learns.

This exercise runs 6 DDPG experiments: 3 seeds with a bug, 3 seeds without. Your job is to diagnose the bug from the learning curves *before* looking at the solution.

Recall the DDPG computation graph:
```python
# Bellman backup (target)
backup  = tf.stop_gradient(r + gamma * (1 - d) * q_pi_targ)
# Losses
pi_loss = -tf.reduce_mean(q_pi)
q_loss  = tf.reduce_mean((q - backup)**2)
```
A bug in actor-critic construction can corrupt what `q_pi`, `q`, and `q_pi_targ` refer to — especially if network weights are accidentally *shared* rather than independent.

In [ ]:
!python /content/spinningup/spinup/exercises/tf1/problem_set_2/exercise2_2.py

In [ ]:
import glob
dirs = sorted(glob.glob('/root/spinningup/data/exercise2_2*/'))
dir_str = ' '.join(dirs)
!python -m spinup.run plot {dir_str}

**Your task — before checking the solution:**
1. Observe the performance gap between bugged and non-bugged runs.
2. Read the actor-critic network creation code in `exercise2_2.py` (not `ddpg/core.py`).
3. Form a hypothesis: what exactly is the bug? How does it break the computation graph?

**Bonus:** Are there hyperparameter settings that would *hide* the effects of this bug?

Official solution: https://spinningup.openai.com/en/latest/spinningup/exercise2_2_soln.html